# Plug-and-Play Model Training

Add audio folders to `AUDIO_FOLDERS`, ensure each has a `{foldername}GT.csv` with `filename` and `label` columns.
The notebook will:
1. Scan all folders, detect/extract missing features (transcripts, text+pause+prosodic, WavLM embeddings, SBERT embeddings)
2. Combine all data with labels
3. Train 5 models: Text XGBoost (baseline), WavLM XGBoost, SBERT XGBoost, Hybrid XGBoost (SBERT+pause+prosodic), Fused XGBoost
4. Stacking meta-learner (WavLM + Hybrid -> LogReg) with threshold sweep + 5-fold CV
5. Save everything to `checkpoints_trained/`

To add new data in the future: just add the folder path to `AUDIO_FOLDERS` and re-run.

In [ ]:
# ================================================================
# CONFIGURATION — just edit this cell
# ================================================================

# List of audio folders — add new ones here and re-run
AUDIO_FOLDERS = [
    r"../audios2",
    r"../audios3",
    r"../audios4",
    r"../audios5",
]

# Train/test split ratio
TEST_RATIO = 0.20
RANDOM_SEED = 42

# Whisper model for transcription (only used if transcripts don't exist)
WHISPER_MODEL = "small"  # "tiny", "base", "small", "medium"

# WavLM settings
WAVLM_MAX_DURATION = 60  # seconds

# Where to save trained models (subfolder inside notebook's directory)
SAVE_DIR = "checkpoints_trained"  # resolved to NB_DIR / checkpoints_trained in imports cell

# Label mapping for GT files
LABEL_MAP = {
    "read": 1, "Read": 1, "READ": 1, "cheating": 1, "Cheating": 1,
    "reading": 1, "Reading": 1, "yes": 1, "Yes": 1, "Y": 1, "1": 1, 1: 1,
    "spontaneous": 0, "Spontaneous": 0, "not cheating": 0, "Not Cheating": 0,
    "Not cheating": 0, "no": 0, "No": 0, "N": 0, "0": 0, 0: 0,
    "genuine": 0, "Genuine": 0,
}

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".wma", ".aac", ".webm", ".mp4"}

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import Counter

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

NB_DIR = Path(".").resolve()

# Resolve SAVE_DIR to be inside NB_DIR
SAVE_DIR = str(NB_DIR / SAVE_DIR)

# Add NB_DIR to path for extract_features_company.py
nb_dir_str = str(NB_DIR)
if nb_dir_str not in sys.path:
    sys.path.insert(0, nb_dir_str)
# Also add parent dir (where extract_features_company.py lives)
parent_str = str(NB_DIR.parent)
if parent_str not in sys.path:
    sys.path.insert(0, parent_str)

print(f"Notebook dir: {NB_DIR}")
print(f"Save dir:     {SAVE_DIR}")
print(f"Folders to process: {len(AUDIO_FOLDERS)}")

## 1. Scan All Folders & Check Existing Features
For each folder, looks for:
- `{foldername}GT.csv` -- ground truth labels (REQUIRED)
- `{foldername}_transcripts.json` -- faster-whisper transcripts
- `{foldername}_features.csv` -- text + pause + prosodic features
- `{foldername}_features_wavlm.csv` -- WavLM embeddings
- `{foldername}_features_sbert.csv` -- SBERT sentence embeddings

In [ ]:
folders = []
total_files = 0
skipped_folders = []

for folder_path in AUDIO_FOLDERS:
    audio_dir = Path(folder_path).resolve()
    folder_name = audio_dir.name

    # Find audio files
    audio_files = sorted([
        f for f in audio_dir.rglob("*")
        if f.suffix.lower() in AUDIO_EXTS and f.is_file()
    ])

    if not audio_files:
        print(f"WARNING: {folder_name} — no audio files found, skipping")
        skipped_folders.append(folder_name)
        continue

    # Check for GT
    gt_path = NB_DIR / f"{folder_name}GT.csv"
    if not gt_path.exists():
        gt_path = audio_dir.parent / f"{folder_name}GT.csv"
    if not gt_path.exists():
        gt_path = audio_dir / f"{folder_name}GT.csv"
    if not gt_path.exists():
        print(f"WARNING: {folder_name} — no GT file found ({folder_name}GT.csv), skipping")
        skipped_folders.append(folder_name)
        continue

    # Check existing artifacts (all in NB_DIR)
    transcripts_json = NB_DIR / f"{folder_name}_transcripts.json"
    features_csv = NB_DIR / f"{folder_name}_features.csv"
    wavlm_csv = NB_DIR / f"{folder_name}_features_wavlm.csv"
    sbert_csv = NB_DIR / f"{folder_name}_features_sbert.csv"

    info = {
        "name": folder_name,
        "audio_dir": audio_dir,
        "audio_files": audio_files,
        "gt_path": gt_path,
        "transcripts_json": transcripts_json,
        "features_csv": features_csv,
        "wavlm_csv": wavlm_csv,
        "sbert_csv": sbert_csv,
        "has_transcripts": transcripts_json.exists(),
        "has_text_features": features_csv.exists(),
        "has_wavlm": wavlm_csv.exists(),
        "has_sbert": sbert_csv.exists(),
    }
    folders.append(info)
    total_files += len(audio_files)

print(f"\n{'='*70}")
print(f"SCAN RESULTS")
print(f"{'='*70}")
print(f"Valid folders: {len(folders)} / {len(AUDIO_FOLDERS)}")
print(f"Total audio files: {total_files}")
if skipped_folders:
    print(f"Skipped: {skipped_folders}")

print(f"\n{'folder':<15s} {'files':>6s} {'GT':>4s} {'trans':>6s} {'text_f':>7s} {'wavlm':>6s} {'sbert':>6s}")
print("-" * 60)
for f in folders:
    print(f"{f['name']:<15s} {len(f['audio_files']):>6d}"
          f"  {'OK':>4s}"
          f"  {'OK' if f['has_transcripts'] else 'NEED':>6s}"
          f"  {'OK' if f['has_text_features'] else 'NEED':>7s}"
          f"  {'OK' if f['has_wavlm'] else 'NEED':>6s}"
          f"  {'OK' if f['has_sbert'] else 'NEED':>6s}")

## 2. Extract Missing Transcripts
Uses faster-whisper with word timestamps. Skipped for folders that already have transcripts.

In [ ]:
need_transcription = [f for f in folders if not f["has_transcripts"]]

# Prompt that nudges Whisper to preserve filler words instead of stripping them
FILLER_PROMPT = "um, uh, hmm, uh-huh, like, you know, I mean, so, actually, basically"

if need_transcription:
    from faster_whisper import WhisperModel
    import librosa

    whisper = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    print(f"Loaded faster-whisper ({WHISPER_MODEL}, int8)")
    print(f"Using filler-preserving prompt: '{FILLER_PROMPT[:60]}...'")

    for folder in need_transcription:
        print(f"\nTranscribing {folder['name']} ({len(folder['audio_files'])} files)...")
        transcripts = {}

        for fp in tqdm(folder["audio_files"], desc=folder["name"]):
            try:
                segments, info = whisper.transcribe(
                    str(fp), language="en", word_timestamps=True,
                    initial_prompt=FILLER_PROMPT,
                )
                words = []
                text_parts = []
                for seg in segments:
                    text_parts.append(seg.text.strip())
                    for w in (seg.words or []):
                        words.append({"word": w.word.strip(), "start": round(w.start, 3), "end": round(w.end, 3)})
                transcripts[fp.name] = {"text": " ".join(text_parts), "words": words}
            except Exception as e:
                print(f"  FAILED: {fp.name}: {e}")
                transcripts[fp.name] = {"text": "", "words": []}

        with open(folder["transcripts_json"], "w", encoding="utf-8") as f:
            json.dump(transcripts, f, indent=2, ensure_ascii=False)
        folder["has_transcripts"] = True
        print(f"  Saved: {folder['transcripts_json'].name}")

        # Also save CSV for VLOOKUP
        csv_path = folder["transcripts_json"].with_suffix(".csv")
        rows = [{"filename": fn, "transcript": data["text"]} for fn, data in transcripts.items()]
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        print(f"  Saved: {csv_path.name}")

    del whisper
    print("\nTranscription complete.")
else:
    print("All folders already have transcripts.")

## 3. Extract Missing Text + Pause + Prosodic Features
Uses `extract_features_company.py` — the same extraction used by the inference pipeline.

In [ ]:
from extract_features_company import (
    compute_text_features, compute_pause_features, compute_prosodic_features,
    _empty_text_features, _empty_pause_features, _empty_prosodic_features,
)

need_text = [f for f in folders if not f["has_text_features"]]

if need_text:
    for folder in need_text:
        print(f"\nExtracting text+pause+prosodic features for {folder['name']}...")

        # Load transcripts
        with open(folder["transcripts_json"], encoding="utf-8") as f:
            transcripts = json.load(f)

        # Load GT for labels
        gt_df = pd.read_csv(folder["gt_path"])
        fn_col = next((c for c in gt_df.columns if c.lower() in ("filename", "file", "name")), gt_df.columns[0])
        label_col = next((c for c in gt_df.columns if c.lower() in ("label", "class", "gt", "ground_truth", "category")), gt_df.columns[-1])
        gt_labels = {}
        for _, row in gt_df.iterrows():
            raw = row[label_col]
            gt_labels[str(row[fn_col])] = LABEL_MAP.get(raw, LABEL_MAP.get(str(raw), -1))

        rows = []
        for fp in tqdm(folder["audio_files"], desc=f"Features ({folder['name']})"):
            fn = fp.name
            entry = transcripts.get(fn, {"text": "", "words": []})

            text_feats = compute_text_features(entry["text"])
            pause_feats = compute_pause_features(entry.get("words", []))
            prosodic_feats = compute_prosodic_features(str(fp))

            row = {"filename": fn, "filepath": str(fp)}
            row.update(text_feats)
            row.update(pause_feats)
            row.update(prosodic_feats)
            row["label_int"] = gt_labels.get(fn, -1)
            row["audio_batch"] = folder["name"]
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(folder["features_csv"], index=False)
        folder["has_text_features"] = True
        print(f"  Saved: {folder['features_csv'].name} ({len(df)} rows, "
              f"{(df['label_int']==1).sum()} cheating, {(df['label_int']==0).sum()} not cheating)")
else:
    print("All folders already have text features.")

## 4. Extract Missing WavLM Embeddings
Uses WavLM-base-plus (768-dim mean-pooled encoder embeddings).

In [ ]:
need_wavlm = [f for f in folders if not f["has_wavlm"]]

if need_wavlm:
    import torch
    import librosa
    from transformers import AutoFeatureExtractor, WavLMModel

    print("Loading WavLM-base-plus...")
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = wavlm_model.eval().to("cpu")
    EMBED_DIM = wavlm_model.config.hidden_size
    SR = 16000

    for folder in need_wavlm:
        print(f"\nExtracting WavLM embeddings for {folder['name']} ({len(folder['audio_files'])} files)...")
        embeddings = []
        filenames = []

        for fp in tqdm(folder["audio_files"], desc=f"WavLM ({folder['name']})"):
            filenames.append(fp.name)
            try:
                audio, _ = librosa.load(str(fp), sr=SR, mono=True, duration=WAVLM_MAX_DURATION)
                if len(audio) < SR:
                    embeddings.append(np.zeros(EMBED_DIM))
                    continue
                with torch.no_grad():
                    inputs = fe(audio, sampling_rate=SR, return_tensors="pt", padding=True)
                    out = wavlm_model(**inputs)
                    emb = out.last_hidden_state.mean(dim=1).squeeze().numpy()
                embeddings.append(emb)
            except Exception as e:
                print(f"  FAILED: {fp.name}: {e}")
                embeddings.append(np.zeros(EMBED_DIM))

        emb_array = np.stack(embeddings)
        cols = [f"wavlm_{i}" for i in range(EMBED_DIM)]
        df_wl = pd.DataFrame(emb_array, columns=cols)
        df_wl.insert(0, "filename", filenames)
        df_wl.to_csv(folder["wavlm_csv"], index=False)
        folder["has_wavlm"] = True
        print(f"  Saved: {folder['wavlm_csv'].name} ({len(df_wl)} rows, {EMBED_DIM} dims)")

    del wavlm_model, fe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\nWavLM extraction complete.")
else:
    print("All folders already have WavLM features.")

## 4b. Extract Missing SBERT Embeddings
Uses `all-MiniLM-L6-v2` (384-dim) to encode transcript text into dense embeddings.
Requires transcripts to exist first. Saves to `{foldername}_features_sbert.csv`.

In [ ]:
need_sbert = [f for f in folders if not f["has_sbert"]]

if need_sbert:
    from sentence_transformers import SentenceTransformer
    sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    SBERT_DIM = 384

    for folder in need_sbert:
        if not folder["has_transcripts"]:
            print(f"Skipping SBERT for {folder['name']} — no transcripts")
            continue

        print(f"\nExtracting SBERT embeddings for {folder['name']}...")
        with open(folder["transcripts_json"], encoding="utf-8") as f:
            transcripts = json.load(f)

        rows = []
        for fp in tqdm(folder["audio_files"], desc=f"SBERT ({folder['name']})"):
            fn = fp.name
            text = transcripts.get(fn, {}).get("text", "")

            if text and len(text.strip()) >= 10:
                emb = sbert_model.encode(text, normalize_embeddings=True)
                row = {"filename": fn}
                row.update({f"sbert_{i}": float(emb[i]) for i in range(SBERT_DIM)})
            else:
                row = {"filename": fn}
                row.update({f"sbert_{i}": 0.0 for i in range(SBERT_DIM)})
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_csv(folder["sbert_csv"], index=False)
        folder["has_sbert"] = True
        print(f"  Saved: {folder['sbert_csv'].name} ({len(df)} rows, {SBERT_DIM} dims)")

    del sbert_model  # free GPU/CPU memory
else:
    print("All folders already have SBERT embeddings.")

## 5. Combine All Data
Merges text features and WavLM embeddings from all folders into unified DataFrames.

In [ ]:
# -- Feature column definitions (must match extraction) --
TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
ALL_TEXT_COLS = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES  # 41 features

# Hybrid feature set: SBERT embeddings + pause + filler + prosodic (no text-statistics)
HYBRID_HANDCRAFTED = ["filler_rate", "filler_count"] + PAUSE_FEATURES + PROSODIC_FEATURES

# -- Load and combine --
all_text_dfs = []
all_wavlm_dfs = []
all_sbert_dfs = []

for folder in folders:
    # Text features
    df_t = pd.read_csv(folder["features_csv"])
    df_t = df_t[df_t["label_int"].isin([0, 1])].reset_index(drop=True)
    if "audio_batch" not in df_t.columns:
        df_t["audio_batch"] = folder["name"]
    all_text_dfs.append(df_t)

    # WavLM features
    df_w = pd.read_csv(folder["wavlm_csv"])
    labeled_filenames = set(df_t["filename"].values)
    df_w = df_w[df_w["filename"].isin(labeled_filenames)].reset_index(drop=True)
    all_wavlm_dfs.append(df_w)

    # SBERT features
    df_s = pd.read_csv(folder["sbert_csv"])
    df_s = df_s[df_s["filename"].isin(labeled_filenames)].reset_index(drop=True)
    all_sbert_dfs.append(df_s)

    print(f"{folder['name']}: {len(df_t)} labeled samples "
          f"({(df_t['label_int']==1).sum()} cheating, {(df_t['label_int']==0).sum()} not cheating)")

# Concatenate
combined_text = pd.concat(all_text_dfs, ignore_index=True)
combined_wavlm = pd.concat(all_wavlm_dfs, ignore_index=True)
combined_sbert = pd.concat(all_sbert_dfs, ignore_index=True)

# Align by filename
combined_wavlm = combined_wavlm.set_index("filename").reindex(combined_text["filename"]).reset_index()
combined_sbert = combined_sbert.set_index("filename").reindex(combined_text["filename"]).reset_index()

y = combined_text["label_int"].values
text_cols = [c for c in ALL_TEXT_COLS if c in combined_text.columns]
wavlm_cols = [c for c in combined_wavlm.columns if c.startswith("wavlm_")]
sbert_cols = [c for c in combined_sbert.columns if c.startswith("sbert_")]
hybrid_handcrafted_cols = [c for c in HYBRID_HANDCRAFTED if c in combined_text.columns]

print(f"\n{'='*60}")
print(f"COMBINED DATASET")
print(f"{'='*60}")
print(f"Total samples: {len(combined_text)}")
print(f"  Cheating:     {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")
print(f"  Not cheating: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"Text features:     {len(text_cols)} (handcrafted)")
print(f"WavLM features:    {len(wavlm_cols)}")
print(f"SBERT features:    {len(sbert_cols)}")
print(f"Hybrid handcrafted: {len(hybrid_handcrafted_cols)} (filler + pause + prosodic)")
print(f"Batches: {combined_text['audio_batch'].value_counts().to_dict()}")

## 6. Train/Test Split
Stratified split preserving class balance. Shows batch distribution in each split.

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=TEST_RATIO, random_state=RANDOM_SEED, stratify=y
)
y_train, y_test = y[idx_train], y[idx_test]

print(f"Train: {len(y_train)} (cheating={y_train.sum()}, not_cheating={(y_train==0).sum()})")
print(f"Test:  {len(y_test)} (cheating={y_test.sum()}, not_cheating={(y_test==0).sum()})")

print(f"\nBatch distribution:")
for name, idxs in [("Train", idx_train), ("Test", idx_test)]:
    counts = combined_text.iloc[idxs]["audio_batch"].value_counts().to_dict()
    print(f"  {name}: {counts}")

## 7. Train Text XGBoost

In [ ]:
X_text = combined_text[text_cols].fillna(0).values

text_scaler = StandardScaler()
X_text_train = text_scaler.fit_transform(X_text[idx_train])
X_text_test = text_scaler.transform(X_text[idx_test])

text_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_SEED,
)

text_model.fit(
    X_text_train, y_train,
    eval_set=[(X_text_train, y_train), (X_text_test, y_test)],
    verbose=False,
)

text_proba_test = text_model.predict_proba(X_text_test)[:, 1]
text_preds_test = (text_proba_test >= 0.5).astype(int)

print(f"Text XGBoost (41 features):")
print(f"  Accuracy:  {accuracy_score(y_test, text_preds_test):.4f}")
print(f"  F1:        {f1_score(y_test, text_preds_test, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, text_preds_test, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, text_preds_test, zero_division=0):.4f}")

## 8. Train WavLM XGBoost

In [ ]:
X_wl = combined_wavlm[wavlm_cols].fillna(0).values

wl_scaler = StandardScaler()
X_wl_train = wl_scaler.fit_transform(X_wl[idx_train])
X_wl_test = wl_scaler.transform(X_wl[idx_test])

wl_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.3,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_SEED,
)

wl_model.fit(
    X_wl_train, y_train,
    eval_set=[(X_wl_train, y_train), (X_wl_test, y_test)],
    verbose=False,
)

wl_proba_test = wl_model.predict_proba(X_wl_test)[:, 1]
wl_preds_test = (wl_proba_test >= 0.5).astype(int)

print(f"WavLM XGBoost (768 features):")
print(f"  Accuracy:  {accuracy_score(y_test, wl_preds_test):.4f}")
print(f"  F1:        {f1_score(y_test, wl_preds_test, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, wl_preds_test, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, wl_preds_test, zero_division=0):.4f}")

## 8b. Train SBERT XGBoost (384-dim sentence embeddings)
Replaces the handcrafted text model with dense sentence embeddings from `all-MiniLM-L6-v2`.
More robust to transcription errors and captures semantic patterns the 41 features miss.

In [ ]:
X_sbert = combined_sbert[sbert_cols].fillna(0).values

sbert_scaler = StandardScaler()
X_sbert_train = sbert_scaler.fit_transform(X_sbert[idx_train])
X_sbert_test = sbert_scaler.transform(X_sbert[idx_test])

sbert_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.3,  # same as WavLM — similar dimensionality
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_SEED,
)

sbert_model.fit(
    X_sbert_train, y_train,
    eval_set=[(X_sbert_train, y_train), (X_sbert_test, y_test)],
    verbose=False,
)

sbert_proba_test = sbert_model.predict_proba(X_sbert_test)[:, 1]
sbert_preds_test = (sbert_proba_test >= 0.5).astype(int)

print(f"SBERT XGBoost ({len(sbert_cols)} features):")
print(f"  Accuracy:  {accuracy_score(y_test, sbert_preds_test):.4f}")
print(f"  F1:        {f1_score(y_test, sbert_preds_test, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, sbert_preds_test, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, sbert_preds_test, zero_division=0):.4f}")

# Compare against handcrafted text model
print(f"\n  vs Text XGBoost (41 features):")
print(f"  Text  -> P={precision_score(y_test, text_preds_test, zero_division=0):.4f}  R={recall_score(y_test, text_preds_test, zero_division=0):.4f}")
print(f"  SBERT -> P={precision_score(y_test, sbert_preds_test, zero_division=0):.4f}  R={recall_score(y_test, sbert_preds_test, zero_division=0):.4f}")

## 8c. Train Hybrid XGBoost (SBERT + Pause + Filler + Prosodic)
Combines SBERT embeddings with the strongest handcrafted features that embeddings cannot capture:
pause features (from timestamps), filler counts, and prosodic features (from audio).
Drops the 18 text-statistic features that SBERT subsumes.

In [ ]:
# Build hybrid feature matrix: SBERT (384) + filler (2) + pause (13) + prosodic (8) = 407
X_hybrid_handcrafted = combined_text[hybrid_handcrafted_cols].fillna(0).values
X_hybrid = np.hstack([X_sbert, X_hybrid_handcrafted])
hybrid_cols = sbert_cols + hybrid_handcrafted_cols

hybrid_scaler = StandardScaler()
X_hybrid_train = hybrid_scaler.fit_transform(X_hybrid[idx_train])
X_hybrid_test = hybrid_scaler.transform(X_hybrid[idx_test])

hybrid_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.25,  # slightly lower for ~407 features
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_SEED,
)

hybrid_model.fit(
    X_hybrid_train, y_train,
    eval_set=[(X_hybrid_train, y_train), (X_hybrid_test, y_test)],
    verbose=False,
)

hybrid_proba_test = hybrid_model.predict_proba(X_hybrid_test)[:, 1]
hybrid_preds_test = (hybrid_proba_test >= 0.5).astype(int)

print(f"Hybrid XGBoost ({len(hybrid_cols)} features = {len(sbert_cols)} SBERT + {len(hybrid_handcrafted_cols)} handcrafted):")
print(f"  Accuracy:  {accuracy_score(y_test, hybrid_preds_test):.4f}")
print(f"  F1:        {f1_score(y_test, hybrid_preds_test, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, hybrid_preds_test, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, hybrid_preds_test, zero_division=0):.4f}")

# Feature importance: SBERT vs handcrafted contribution
importances = hybrid_model.feature_importances_
sbert_imp = importances[:len(sbert_cols)].sum()
hc_imp = importances[len(sbert_cols):].sum()
total_imp = sbert_imp + hc_imp
print(f"\n  Feature importance: SBERT={sbert_imp/total_imp*100:.1f}%, handcrafted={hc_imp/total_imp*100:.1f}%")

# Show top 10 handcrafted features by importance
hc_importances = list(zip(hybrid_handcrafted_cols, importances[len(sbert_cols):]))
hc_importances.sort(key=lambda x: x[1], reverse=True)
print(f"\n  Top handcrafted features:")
for name, imp in hc_importances[:10]:
    print(f"    {name}: {imp:.4f}")

## 9. Train Fused XGBoost (Text + WavLM Concatenated)

In [ ]:
X_fused = np.hstack([X_text, X_wl])  # 41 + 768 = 809 features
all_fused_cols = text_cols + wavlm_cols

fused_scaler = StandardScaler()
X_fused_train = fused_scaler.fit_transform(X_fused[idx_train])
X_fused_test = fused_scaler.transform(X_fused[idx_test])

fused_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.2,  # low — 809 features, prevent overfitting
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_SEED,
)

fused_model.fit(
    X_fused_train, y_train,
    eval_set=[(X_fused_train, y_train), (X_fused_test, y_test)],
    verbose=False,
)

fused_proba_test = fused_model.predict_proba(X_fused_test)[:, 1]
fused_preds_test = (fused_proba_test >= 0.5).astype(int)

print(f"Fused XGBoost ({len(all_fused_cols)} features):")
print(f"  Accuracy:  {accuracy_score(y_test, fused_preds_test):.4f}")
print(f"  F1:        {f1_score(y_test, fused_preds_test, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, fused_preds_test, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, fused_preds_test, zero_division=0):.4f}")

# Feature importance: text vs wavlm contribution
importances = fused_model.feature_importances_
text_imp = importances[:len(text_cols)].sum()
wavlm_imp = importances[len(text_cols):].sum()
total_imp = text_imp + wavlm_imp
print(f"\n  Feature importance: text={text_imp/total_imp*100:.1f}%, wavlm={wavlm_imp/total_imp*100:.1f}%")

## 10. Stacking Meta-Learner (replaces manual weight search)
Instead of grid-searching weights, train a logistic regression on out-of-fold predictions
from the two best base models (WavLM XGBoost + Hybrid XGBoost).
This automatically learns optimal combination weights AND calibrates probabilities.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# -- Summary of all individual models on test set --
print("="*60)
print("ALL MODELS — TEST SET COMPARISON")
print("="*60)

models_summary = {
    "Text (41 feat)": text_preds_test,
    "WavLM (768 feat)": wl_preds_test,
    "Fused (809 feat)": fused_preds_test,
    "SBERT (384 feat)": sbert_preds_test,
    "Hybrid (407 feat)": hybrid_preds_test,
}
for name, preds in models_summary.items():
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f"  {name:<22s}  F1={f:.4f}  P={p:.4f}  R={r:.4f}")

# -- Stacking: OOF predictions from WavLM + Hybrid --
print(f"\n{'='*60}")
print("STACKING META-LEARNER (WavLM + Hybrid)")
print("="*60)

skf_stack = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
X_hybrid_all = np.hstack([
    combined_sbert[sbert_cols].fillna(0).values,
    combined_text[hybrid_handcrafted_cols].fillna(0).values,
])
X_wl_all = combined_wavlm[wavlm_cols].fillna(0).values

oof_wavlm = np.zeros(len(y))
oof_hybrid = np.zeros(len(y))

for fold, (tr_idx, te_idx) in enumerate(skf_stack.split(X_hybrid_all, y)):
    y_tr = y[tr_idx]
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    # WavLM
    sc_w = StandardScaler()
    Xw_tr = sc_w.fit_transform(X_wl_all[tr_idx])
    Xw_te = sc_w.transform(X_wl_all[te_idx])
    m_w = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.3, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_w.fit(Xw_tr, y_tr, eval_set=[(Xw_te, y[te_idx])], verbose=False)
    oof_wavlm[te_idx] = m_w.predict_proba(Xw_te)[:, 1]

    # Hybrid
    sc_h = StandardScaler()
    Xh_tr = sc_h.fit_transform(X_hybrid_all[tr_idx])
    Xh_te = sc_h.transform(X_hybrid_all[te_idx])
    m_h = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.25, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_h.fit(Xh_tr, y_tr, eval_set=[(Xh_te, y[te_idx])], verbose=False)
    oof_hybrid[te_idx] = m_h.predict_proba(Xh_te)[:, 1]

    print(f"  Fold {fold+1}: WavLM OOF mean={oof_wavlm[te_idx].mean():.3f}, Hybrid OOF mean={oof_hybrid[te_idx].mean():.3f}")

# Train meta-learner on OOF predictions
meta_X = np.column_stack([oof_wavlm, oof_hybrid])
meta_model = LogisticRegression(C=1.0, random_state=RANDOM_SEED)
meta_model.fit(meta_X, y)

print(f"\nMeta-learner coefficients:")
print(f"  WavLM weight:  {meta_model.coef_[0][0]:.4f}")
print(f"  Hybrid weight: {meta_model.coef_[0][1]:.4f}")
print(f"  Intercept:     {meta_model.intercept_[0]:.4f}")

# OOF performance at various thresholds
meta_proba_oof = meta_model.predict_proba(meta_X)[:, 1]

print(f"\n{'='*60}")
print("THRESHOLD SWEEP (on OOF predictions)")
print("="*60)

rows = []
for thr in np.arange(0.20, 0.81, 0.05):
    preds = (meta_proba_oof >= thr).astype(int)
    p = precision_score(y, preds, zero_division=0)
    r = recall_score(y, preds, zero_division=0)
    f = f1_score(y, preds, zero_division=0)
    rows.append({"threshold": round(thr, 2), "precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4)})

thr_df = pd.DataFrame(rows)
print(thr_df.to_string(index=False))

# Best by F1
best_f1_row = thr_df.loc[thr_df["f1"].idxmax()]
# Best precision with recall >= 0.80
prec_filtered = thr_df[thr_df["recall"] >= 0.80]
best_prec_row = prec_filtered.loc[prec_filtered["precision"].idxmax()] if len(prec_filtered) > 0 else best_f1_row
# Best precision with recall >= 0.50 (fallback)
prec_filtered_50 = thr_df[thr_df["recall"] >= 0.50]
best_prec_row_50 = prec_filtered_50.loc[prec_filtered_50["precision"].idxmax()] if len(prec_filtered_50) > 0 else best_f1_row

print(f"\nBest F1: thr={best_f1_row['threshold']} -> P={best_f1_row['precision']}, R={best_f1_row['recall']}, F1={best_f1_row['f1']}")
print(f"Best P (R>=0.80): thr={best_prec_row['threshold']} -> P={best_prec_row['precision']}, R={best_prec_row['recall']}, F1={best_prec_row['f1']}")
print(f"Best P (R>=0.50): thr={best_prec_row_50['threshold']} -> P={best_prec_row_50['precision']}, R={best_prec_row_50['recall']}, F1={best_prec_row_50['f1']}")

BEST_THRESHOLD = float(best_f1_row["threshold"])

# PR curve
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
ax.plot(thr_df["threshold"], thr_df["precision"], "b-o", label="Precision", markersize=4)
ax.plot(thr_df["threshold"], thr_df["recall"], "r-o", label="Recall", markersize=4)
ax.plot(thr_df["threshold"], thr_df["f1"], "g--o", label="F1", markersize=4)
ax.axvline(x=BEST_THRESHOLD, color="gray", linestyle="--", alpha=0.5, label=f"Best thr={BEST_THRESHOLD}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Stacked Ensemble — Threshold Sweep (OOF)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(NB_DIR / "train_stacked_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. 5-Fold Cross-Validation (All Models)
Compares all architectures: Text, WavLM, Fused, SBERT, Hybrid, and Stacked ensemble.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_results = []

X_text_all = combined_text[text_cols].fillna(0).values
X_wl_all = combined_wavlm[wavlm_cols].fillna(0).values
X_sbert_all = combined_sbert[sbert_cols].fillna(0).values
X_hybrid_all = np.hstack([X_sbert_all, combined_text[hybrid_handcrafted_cols].fillna(0).values])
X_fused_all = np.hstack([X_text_all, X_wl_all])

print(f"5-Fold CV — Comparing all models (threshold={BEST_THRESHOLD})")
print("-" * 90)

for fold, (tr_idx, te_idx) in enumerate(skf.split(X_text_all, y)):
    y_tr, y_te = y[tr_idx], y[te_idx]
    spw = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    # Text model
    sc_t = StandardScaler()
    Xt_tr = sc_t.fit_transform(X_text_all[tr_idx])
    Xt_te = sc_t.transform(X_text_all[te_idx])
    m_t = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_t.fit(Xt_tr, y_tr, eval_set=[(Xt_te, y_te)], verbose=False)

    # WavLM model
    sc_w = StandardScaler()
    Xw_tr = sc_w.fit_transform(X_wl_all[tr_idx])
    Xw_te = sc_w.transform(X_wl_all[te_idx])
    m_w = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.3, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_w.fit(Xw_tr, y_tr, eval_set=[(Xw_te, y_te)], verbose=False)

    # Fused model (text + wavlm)
    sc_f = StandardScaler()
    Xf_tr = sc_f.fit_transform(X_fused_all[tr_idx])
    Xf_te = sc_f.transform(X_fused_all[te_idx])
    m_f = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.2, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_f.fit(Xf_tr, y_tr, eval_set=[(Xf_te, y_te)], verbose=False)

    # SBERT model
    sc_s = StandardScaler()
    Xs_tr = sc_s.fit_transform(X_sbert_all[tr_idx])
    Xs_te = sc_s.transform(X_sbert_all[te_idx])
    m_s = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.3, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_s.fit(Xs_tr, y_tr, eval_set=[(Xs_te, y_te)], verbose=False)

    # Hybrid model (SBERT + pause + filler + prosodic)
    sc_h = StandardScaler()
    Xh_tr = sc_h.fit_transform(X_hybrid_all[tr_idx])
    Xh_te = sc_h.transform(X_hybrid_all[te_idx])
    m_h = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.25, min_child_weight=3,
                             scale_pos_weight=spw, eval_metric="logloss",
                             early_stopping_rounds=30, random_state=RANDOM_SEED)
    m_h.fit(Xh_tr, y_tr, eval_set=[(Xh_te, y_te)], verbose=False)

    # Stacked ensemble (WavLM + Hybrid -> LogReg)
    w_prob = m_w.predict_proba(Xw_te)[:, 1]
    h_prob = m_h.predict_proba(Xh_te)[:, 1]
    stack_X_te = np.column_stack([w_prob, h_prob])
    stack_pred_proba = meta_model.predict_proba(stack_X_te)[:, 1]
    stack_pred = (stack_pred_proba >= BEST_THRESHOLD).astype(int)

    # Collect results
    result = {"fold": fold + 1}
    for name, preds in [("text", (m_t.predict_proba(Xt_te)[:, 1] >= 0.5).astype(int)),
                         ("wavlm", (w_prob >= 0.5).astype(int)),
                         ("fused", (m_f.predict_proba(Xf_te)[:, 1] >= 0.5).astype(int)),
                         ("sbert", (m_s.predict_proba(Xs_te)[:, 1] >= 0.5).astype(int)),
                         ("hybrid", (h_prob >= 0.5).astype(int)),
                         ("stacked", stack_pred)]:
        result[f"{name}_f1"] = f1_score(y_te, preds, zero_division=0)
        result[f"{name}_prec"] = precision_score(y_te, preds, zero_division=0)
        result[f"{name}_recall"] = recall_score(y_te, preds, zero_division=0)

    fold_results.append(result)
    print(f"  Fold {fold+1}: "
          f"text F1={result['text_f1']:.3f}  "
          f"wavlm={result['wavlm_f1']:.3f}  "
          f"sbert={result['sbert_f1']:.3f}  "
          f"hybrid={result['hybrid_f1']:.3f}  "
          f"stacked={result['stacked_f1']:.3f}")

fold_df = pd.DataFrame(fold_results)
print(f"\n{'='*90}")
print(f"MEAN (5 folds):")
for name in ["text", "wavlm", "fused", "sbert", "hybrid", "stacked"]:
    f1_m = fold_df[f'{name}_f1'].mean()
    f1_s = fold_df[f'{name}_f1'].std()
    p_m = fold_df[f'{name}_prec'].mean()
    r_m = fold_df[f'{name}_recall'].mean()
    print(f"  {name:<10s}: F1={f1_m:.4f} +/- {f1_s:.4f}  P={p_m:.4f}  R={r_m:.4f}")

## 12. Save Models

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

# Text model (baseline)
text_model.save_model(f"{SAVE_DIR}/xgboost_text.json")
joblib.dump(text_scaler, f"{SAVE_DIR}/scaler_text.pkl")

# WavLM model
wl_model.save_model(f"{SAVE_DIR}/xgboost_wavlm.json")
joblib.dump(wl_scaler, f"{SAVE_DIR}/scaler_wavlm.pkl")

# Fused model (baseline)
fused_model.save_model(f"{SAVE_DIR}/xgboost_fused.json")
joblib.dump(fused_scaler, f"{SAVE_DIR}/scaler_fused.pkl")

# SBERT model
sbert_model.save_model(f"{SAVE_DIR}/xgboost_sbert.json")
joblib.dump(sbert_scaler, f"{SAVE_DIR}/scaler_sbert.pkl")

# Hybrid model (SBERT + pause + filler + prosodic)
hybrid_model.save_model(f"{SAVE_DIR}/xgboost_hybrid.json")
joblib.dump(hybrid_scaler, f"{SAVE_DIR}/scaler_hybrid.pkl")

# Stacking meta-learner
joblib.dump(meta_model, f"{SAVE_DIR}/meta_logistic.pkl")

# Config / metadata
config = {
    "audio_folders": [f["name"] for f in folders],
    "n_samples": len(y),
    "n_cheating": int((y == 1).sum()),
    "n_not_cheating": int((y == 0).sum()),
    "n_train": len(idx_train),
    "n_test": len(idx_test),
    "text_feature_columns": text_cols,
    "wavlm_feature_columns": wavlm_cols,
    "sbert_feature_columns": sbert_cols,
    "hybrid_handcrafted_columns": hybrid_handcrafted_cols,
    "hybrid_feature_columns": hybrid_cols,
    "fused_feature_columns": all_fused_cols,
    "architecture": "stacked (WavLM XGB + Hybrid XGB -> LogReg)",
    "meta_learner_coefs": {
        "wavlm": float(meta_model.coef_[0][0]),
        "hybrid": float(meta_model.coef_[0][1]),
        "intercept": float(meta_model.intercept_[0]),
    },
    "threshold": BEST_THRESHOLD,
    "test_metrics": {
        "text": {"f1": round(f1_score(y_test, text_preds_test, zero_division=0), 4),
                 "precision": round(precision_score(y_test, text_preds_test, zero_division=0), 4),
                 "recall": round(recall_score(y_test, text_preds_test, zero_division=0), 4)},
        "wavlm": {"f1": round(f1_score(y_test, wl_preds_test, zero_division=0), 4),
                   "precision": round(precision_score(y_test, wl_preds_test, zero_division=0), 4),
                   "recall": round(recall_score(y_test, wl_preds_test, zero_division=0), 4)},
        "sbert": {"f1": round(f1_score(y_test, sbert_preds_test, zero_division=0), 4),
                  "precision": round(precision_score(y_test, sbert_preds_test, zero_division=0), 4),
                  "recall": round(recall_score(y_test, sbert_preds_test, zero_division=0), 4)},
        "hybrid": {"f1": round(f1_score(y_test, hybrid_preds_test, zero_division=0), 4),
                   "precision": round(precision_score(y_test, hybrid_preds_test, zero_division=0), 4),
                   "recall": round(recall_score(y_test, hybrid_preds_test, zero_division=0), 4)},
        "fused": {"f1": round(f1_score(y_test, fused_preds_test, zero_division=0), 4),
                   "precision": round(precision_score(y_test, fused_preds_test, zero_division=0), 4),
                   "recall": round(recall_score(y_test, fused_preds_test, zero_division=0), 4)},
    },
    "cv_metrics": {name: {
        "f1_mean": round(fold_df[f'{name}_f1'].mean(), 4),
        "f1_std": round(fold_df[f'{name}_f1'].std(), 4),
        "prec_mean": round(fold_df[f'{name}_prec'].mean(), 4),
        "recall_mean": round(fold_df[f'{name}_recall'].mean(), 4),
    } for name in ["text", "wavlm", "fused", "sbert", "hybrid", "stacked"]},
    "batch_distribution": combined_text["audio_batch"].value_counts().to_dict(),
}

with open(f"{SAVE_DIR}/training_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}/:")
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f"  {fn}")

print(f"\n{'='*60}")
print(f"TO USE IN inference_playground.ipynb, update config cell:")
print(f"{'='*60}")
print(f'WAVLM_XGBOOST_MODEL = r"{SAVE_DIR}/xgboost_wavlm.json"')
print(f'WAVLM_SCALER        = r"{SAVE_DIR}/scaler_wavlm.pkl"')
print(f'HYBRID_XGBOOST_MODEL = r"{SAVE_DIR}/xgboost_hybrid.json"')
print(f'HYBRID_SCALER        = r"{SAVE_DIR}/scaler_hybrid.pkl"')
print(f'META_MODEL           = r"{SAVE_DIR}/meta_logistic.pkl"')
print(f'THRESHOLD  = {BEST_THRESHOLD}')

## 13. Detailed Test Set Results

In [ ]:
# Full classification reports — Stacked ensemble
stack_test_X = np.column_stack([wl_proba_test, hybrid_proba_test])
stack_proba_test = meta_model.predict_proba(stack_test_X)[:, 1]
stack_preds_test = (stack_proba_test >= BEST_THRESHOLD).astype(int)

print("STACKED ENSEMBLE (WavLM + Hybrid -> LogReg)")
print(f"Threshold: {BEST_THRESHOLD}")
print(classification_report(y_test, stack_preds_test, target_names=["not cheating", "cheating"]))
cm = confusion_matrix(y_test, stack_preds_test, labels=[0, 1])
print(f"TN={cm[0,0]}  FP={cm[0,1]}")
print(f"FN={cm[1,0]}  TP={cm[1,1]}")

# Compare all models side by side
print(f"\n{'='*60}")
print("ALL MODELS — TEST SET COMPARISON")
print(f"{'='*60}")
all_models = {
    "Text (41)": text_preds_test,
    "WavLM (768)": wl_preds_test,
    "Fused (809)": fused_preds_test,
    "SBERT (384)": sbert_preds_test,
    "Hybrid (407)": hybrid_preds_test,
    "Stacked": stack_preds_test,
}
for name, preds in all_models.items():
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    a = accuracy_score(y_test, preds)
    print(f"  {name:<16s}  Acc={a:.4f}  F1={f:.4f}  P={p:.4f}  R={r:.4f}")

# Per-batch performance (stacked ensemble)
print(f"\n{'='*60}")
print("PER-BATCH PERFORMANCE (stacked ensemble)")
print(f"{'='*60}")

test_df = combined_text.iloc[idx_test].copy()
test_df["y_true"] = y_test
test_df["y_pred_stacked"] = stack_preds_test
test_df["y_pred_hybrid"] = hybrid_preds_test
test_df["y_pred_text"] = text_preds_test

for batch in sorted(test_df["audio_batch"].unique()):
    batch_data = test_df[test_df["audio_batch"] == batch]
    yt = batch_data["y_true"].values
    ys = batch_data["y_pred_stacked"].values
    yh = batch_data["y_pred_hybrid"].values
    ytx = batch_data["y_pred_text"].values
    print(f"\n{batch} (n={len(batch_data)}, cheating={yt.sum()}, not_cheating={(yt==0).sum()}):")
    print(f"  Stacked: F1={f1_score(yt, ys, zero_division=0):.4f}  "
          f"P={precision_score(yt, ys, zero_division=0):.4f}  "
          f"R={recall_score(yt, ys, zero_division=0):.4f}")
    print(f"  Hybrid:  F1={f1_score(yt, yh, zero_division=0):.4f}  "
          f"P={precision_score(yt, yh, zero_division=0):.4f}  "
          f"R={recall_score(yt, yh, zero_division=0):.4f}")
    print(f"  Text:    F1={f1_score(yt, ytx, zero_division=0):.4f}  "
          f"P={precision_score(yt, ytx, zero_division=0):.4f}  "
          f"R={recall_score(yt, ytx, zero_division=0):.4f}")